<a href="https://www.kaggle.com/code/simarbirsinghsandhu/catboost-gpu-pb-0-94964?scriptVersionId=316047397" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# 🏎️ F1 Pit Stop Prediction | CatBoost GPU Baseline
### Playground Series S6E5

**🏆 OOF ROC-AUC: 0.957893 | 📊 Public LB: 0.94964**

⚡ CatBoost + Optuna (GPU) | 5-Fold Stratified CV

**Why CatBoost over XGBoost here:**
- Handles `Driver`, `Compound`, `Race` natively — no label encoding needed
- Built-in ordered boosting reduces overfitting on synthetic data
- Different error patterns from XGBoost → blend both for higher LB

> Upvote if useful 🙌 | [XGBoost Baseline](https://www.kaggle.com/code/simarbirsinghsandhu/xgboost-gpu-fe-encoding) | [Full EDA](https://www.kaggle.com/code/simarbirsinghsandhu/s6e5-detailed-eda-original-dataset-analysis)

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import catboost as cb
import optuna
import matplotlib.pyplot as plt
import warnings

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
print(f"catboost {cb.__version__}")

catboost 1.2.10


## 2. Config

In [2]:
CONFIG = {
    "train_path"       : "/kaggle/input/competitions/playground-series-s6e5/train.csv",
    "test_path"        : "/kaggle/input/competitions/playground-series-s6e5/test.csv",
    "orig_path"        : "/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv",
    "target_col"       : "PitNextLap",
    "id_col"           : "id",
    "n_folds"          : 5,
    "seed"             : 42,
    "optuna_trials"    : 50,
    "optuna_folds"     : 3,
    "optuna_subsample" : 200_000,
}

## 3. Load Data

In [3]:
train = pd.read_csv(CONFIG["train_path"])
test  = pd.read_csv(CONFIG["test_path"])
orig  = pd.read_csv(CONFIG["orig_path"])

orig.drop(columns=["Normalized_TyreLife"], inplace=True, errors="ignore")

print(f"Train : {train.shape}")
print(f"Test  : {test.shape}")
print(f"Orig  : {orig.shape}")
print(f"\nTarget distribution:")
print(train["PitNextLap"].value_counts(normalize=True).round(4))

Train : (439140, 16)
Test  : (188165, 15)
Orig  : (101371, 15)

Target distribution:
PitNextLap
0.0    0.801
1.0    0.199
Name: proportion, dtype: float64


## 4. Feature Engineering

Same features as the XGBoost baseline — built from EDA findings.  
Key difference: `Driver`, `Compound`, `Race` passed as **categorical features** to CatBoost directly. No label encoding needed — CatBoost handles them natively with its own ordered target statistics, which is more principled than simple label encoding.

In [4]:
COMPOUND_STINT_MEDIANS = {
    "SOFT"        : 14.0,
    "MEDIUM"      : 17.0,
    "HARD"        : 23.0,
    "INTERMEDIATE": 16.0,
    "WET"         : 14.0,
}

COMPOUND_HARDNESS = {
    "SOFT"        : 1,
    "MEDIUM"      : 2,
    "HARD"        : 3,
    "INTERMEDIATE": 1,
    "WET"         : 0,
}

def engineer_features(df):
    df = df.copy()

    df["ExpectedStint"]       = df["Compound"].map(COMPOUND_STINT_MEDIANS).fillna(17.0)
    df["TyreLife_Normalized"] = df["TyreLife"] / df["ExpectedStint"]
    df["TyreLife_sq"]         = df["TyreLife"] ** 2
    df["TyreLife_sqrt"]       = np.sqrt(df["TyreLife"])
    df["TyreLife_log1p"]      = np.log1p(df["TyreLife"])
    df["Compound_Hardness"]   = df["Compound"].map(COMPOUND_HARDNESS).fillna(2).astype(int)
    df["TyreLife_x_Hardness"] = df["TyreLife"] * df["Compound_Hardness"]
    df["Norm_x_Hardness"]     = df["TyreLife_Normalized"] * df["Compound_Hardness"]
    df["Is_Fresh"]            = (df["TyreLife"] <= 3).astype(np.int8)
    df["Is_Old"]              = (df["TyreLife"] > 20).astype(np.int8)
    df["Is_VeryOld"]          = (df["TyreLife"] > 40).astype(np.int8)
    df["Is_FirstStint"]       = (df["Stint"] == 1).astype(np.int8)
    df["DegRate"]             = df["Cumulative_Degradation"] / (df["TyreLife"] + 1)
    df["Early_Race"]          = (df["RaceProgress"] < 0.25).astype(np.int8)
    df["Late_Race"]           = (df["RaceProgress"] >= 0.75).astype(np.int8)
    df["VeryLate_Race"]       = (df["RaceProgress"] >= 0.90).astype(np.int8)
    df["Is_2025"]             = (df["Year"] == 2025).astype(np.int8)

    return df

train = engineer_features(train)
test  = engineer_features(test)
orig  = engineer_features(orig)

print(f"Engineered 16 new features")

Engineered 16 new features


## 5. Preprocessing

In [5]:
CAT_COLS  = ["Driver", "Compound", "Race"]
DROP_COLS = ["id", "PitNextLap", "ExpectedStint"]

# Convert categoricals to string for CatBoost
for col in CAT_COLS:
    train[col] = train[col].astype(str)
    test[col]  = test[col].astype(str)
    orig[col]  = orig[col].astype(str)

# Domain gap flag
train["is_original"] = 0
orig["is_original"]  = 1
test["is_original"]  = 0

FEATURE_COLS = [c for c in train.columns if c not in DROP_COLS]

# Align original
orig_aligned = orig.reindex(columns=FEATURE_COLS + ["PitNextLap"], fill_value=0)
orig_aligned["PitNextLap"] = orig_aligned["PitNextLap"].astype(float)
train_full = pd.concat([train, orig_aligned], ignore_index=True)

X_df      = train_full[FEATURE_COLS]
y         = train_full["PitNextLap"].values
X_test_df = test[FEATURE_COLS]

# CatBoost needs categorical feature indices
cat_feature_indices = [X_df.columns.get_loc(c) for c in CAT_COLS]

print(f"Features         : {len(FEATURE_COLS)}")
print(f"Train rows       : {len(X_df):,}")
print(f"Positive rate    : {y.mean():.4f}")
print(f"Cat feature idx  : {cat_feature_indices}")

Features         : 31
Train rows       : 540,511
Positive rate    : 0.2094
Cat feature idx  : [0, 1, 2]


## 6. Optuna Hyperparameter Search

3-fold CV on 200K subsample, optimizing ROC-AUC directly.  
CatBoost search space differs from XGBoost — `depth`, `l2_leaf_reg`, `bagging_temperature` are the key levers.

In [6]:
def run_optuna(X_df, y, cat_indices, cfg):
    n_sub = cfg["optuna_subsample"]
    rng   = np.random.RandomState(cfg["seed"])
    idx   = rng.choice(len(X_df), min(n_sub, len(X_df)), replace=False)
    X_opt = X_df.iloc[idx].reset_index(drop=True)
    y_opt = y[idx]
    print(f"Subsampled to {len(X_opt):,} rows for Optuna")

    kf = StratifiedKFold(n_splits=cfg["optuna_folds"],
                         shuffle=True, random_state=cfg["seed"])

    def objective(trial):
        params = {
            "iterations"          : 3000,
            "learning_rate"       : trial.suggest_float("learning_rate",       0.01,  0.15, log=True),
            "depth"               : trial.suggest_int(  "depth",               4,     8),
            "l2_leaf_reg"         : trial.suggest_float("l2_leaf_reg",         1.0,   10.0, log=True),
            "bagging_temperature" : trial.suggest_float("bagging_temperature", 0.0,   1.0),
            "random_strength"     : trial.suggest_float("random_strength",     0.0,   2.0),
            "border_count"        : trial.suggest_int(  "border_count",        32,    255),
            "scale_pos_weight"    : trial.suggest_float("scale_pos_weight",    1.0,   8.0),
            "task_type"           : "GPU",
            "eval_metric"         : "Logloss",
            "loss_function"       : "Logloss",
            "random_seed"         : cfg["seed"],
            "verbose"             : False,
        }

        aucs = []
        for tr_idx, va_idx in kf.split(X_opt, y_opt):
            X_tr = X_opt.iloc[tr_idx]
            X_va = X_opt.iloc[va_idx]
            y_tr = y_opt[tr_idx]
            y_va = y_opt[va_idx]

            train_pool = cb.Pool(X_tr, y_tr, cat_features=cat_indices)
            val_pool   = cb.Pool(X_va, y_va, cat_features=cat_indices)

            model = cb.CatBoostClassifier(**params)
            model.fit(train_pool,
                      eval_set=val_pool,
                      early_stopping_rounds=100,
                      verbose=False)

            aucs.append(roc_auc_score(y_va,
                        model.predict_proba(X_va)[:, 1]))
        return np.mean(aucs)

    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=cfg["seed"]))
    study.optimize(objective, n_trials=cfg["optuna_trials"], show_progress_bar=True)

    print(f"\n✅ Best ROC-AUC : {study.best_value:.6f}")
    print(f"Best params    :\n{study.best_params}")
    return study.best_params, study
    
# Uncomment to run optuna tune
# best_params, study = run_optuna(X_df, y, cat_feature_indices, CONFIG)

## 7. Best params
These are auto-filled by Optuna above. If you want to skip Optuna on re-runs, comment out the cell above and paste your best params here.

In [7]:
# best_params is already set by Optuna above
# If you skipped Optuna, uncomment and paste your params

best_params = {
    "learning_rate"     : 0.15,
    "depth"             : 6,
    "l2_leaf_reg"       : 3.0,
    "bootstrap_type"    : "Bernoulli",
    "subsample"         : 0.8,
    "random_strength"   : 1.0,
    "border_count"      : 128,
    "scale_pos_weight"  : 4.0,
}

print("Using params:", best_params)

Using params: {'learning_rate': 0.15, 'depth': 6, 'l2_leaf_reg': 3.0, 'bootstrap_type': 'Bernoulli', 'subsample': 0.8, 'random_strength': 1.0, 'border_count': 128, 'scale_pos_weight': 4.0}


## 8. OOF Training

Full 5-fold stratified CV. CatBoost Pool objects handle categorical features properly inside each fold.

In [8]:
def catboost_oof(X_df, y, X_test_df, cat_indices, params, cfg):
    kf         = StratifiedKFold(n_splits=cfg["n_folds"],
                                 shuffle=True, random_state=cfg["seed"])
    oof_preds  = np.zeros(len(X_df))
    test_preds = np.zeros(len(X_test_df))
    fold_aucs  = []

    run_params = {
        "iterations"          : 7000,
        "task_type"           : "GPU",
        "eval_metric"         : "Logloss",
        "loss_function"       : "Logloss",
        "random_seed"         : cfg["seed"],
        "verbose"             : 200,
        **params,
    }

    test_pool = cb.Pool(X_test_df, cat_features=cat_indices)

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_df, y)):
        X_tr = X_df.iloc[tr_idx]
        X_va = X_df.iloc[va_idx]
        y_tr = y[tr_idx]
        y_va = y[va_idx]

        train_pool = cb.Pool(X_tr, y_tr, cat_features=cat_indices)
        val_pool   = cb.Pool(X_va, y_va, cat_features=cat_indices)

        model = cb.CatBoostClassifier(**run_params)
        model.fit(train_pool,
                  eval_set=val_pool,
                  early_stopping_rounds=100,
                  verbose=500)

        val_preds         = model.predict_proba(X_va)[:, 1]
        oof_preds[va_idx] = val_preds
        test_preds       += model.predict_proba(X_test_df)[:, 1] / cfg["n_folds"]
        auc               = roc_auc_score(y_va, val_preds)
        fold_aucs.append(auc)
        print(f"Fold {fold+1} AUC: {auc:.6f}\n")

    oof_auc = roc_auc_score(y, oof_preds)
    print("─" * 45)
    for i, s in enumerate(fold_aucs):
        print(f"  Fold {i+1}: {s:.6f}")
    print("─" * 45)
    print(f"  Mean  : {np.mean(fold_aucs):.6f} ± {np.std(fold_aucs):.6f}")
    print(f"  OOF   : {oof_auc:.6f}")
    print("─" * 45)

    return oof_preds, test_preds, fold_aucs, oof_auc, model

oof_preds, test_preds, fold_aucs, oof_auc, last_model = catboost_oof(
    X_df, y, X_test_df, cat_feature_indices, best_params, CONFIG
)

0:	learn: 0.5938888	test: 0.5939930	best: 0.5939930 (0)	total: 6.52s	remaining: 12h 41m 5s
500:	learn: 0.2607366	test: 0.2691176	best: 0.2691176 (500)	total: 22.3s	remaining: 4m 49s
1000:	learn: 0.2416957	test: 0.2620254	best: 0.2620254 (1000)	total: 38.2s	remaining: 3m 48s
1500:	learn: 0.2287936	test: 0.2597076	best: 0.2596668 (1474)	total: 54.2s	remaining: 3m 18s
2000:	learn: 0.2179141	test: 0.2581357	best: 0.2581314 (1998)	total: 1m 10s	remaining: 2m 55s
bestTest = 0.2578830006
bestIteration = 2101
Shrink model to first 2102 iterations.
Fold 1 AUC: 0.958034

0:	learn: 0.5939751	test: 0.5943060	best: 0.5943060 (0)	total: 36.4ms	remaining: 4m 15s
500:	learn: 0.2591389	test: 0.2716857	best: 0.2716857 (500)	total: 15.8s	remaining: 3m 24s
1000:	learn: 0.2412299	test: 0.2648961	best: 0.2648961 (1000)	total: 31.7s	remaining: 3m 10s
1500:	learn: 0.2281807	test: 0.2622307	best: 0.2622307 (1500)	total: 47.7s	remaining: 2m 54s
bestTest = 0.261088812
bestIteration = 1858
Shrink model to first 1

## 9. Submission

In [9]:
submission = pd.DataFrame({
    "id"        : test["id"],
    "PitNextLap": test_preds,
})
submission.to_csv("submission.csv", index=False)

print("✅ Submission saved")
print(f"   Shape  : {submission.shape}")
print(f"   Min    : {test_preds.min():.4f}")
print(f"   Max    : {test_preds.max():.4f}")
print(f"   Mean   : {test_preds.mean():.4f}")
print(f"\n   OOF AUC: {oof_auc:.6f}")
print(submission.head())

✅ Submission saved
   Shape  : (188165, 2)
   Min    : 0.0000
   Max    : 0.9986
   Mean   : 0.2866

   OOF AUC: 0.957874
       id  PitNextLap
0  439140    0.026048
1  439141    0.025381
2  439142    0.015851
3  439143    0.304467
4  439144    0.964738


## Summary

| | Score |
|---|---|
| OOF ROC-AUC | 0.957893 |
| Public LB | 0.94964 |

**Why CatBoost complements XGBoost:**
- Native categorical handling → Driver/Race/Compound encoded more carefully
- Ordered boosting → different bias-variance tradeoff
- Different prediction errors → blending both improves LB

**Next step — blend XGBoost + CatBoost OOFs:**
```python
# Simple rank blend
from scipy.stats import rankdata
blend = 0.5 * rankdata(xgb_preds)/len(xgb_preds) + \
        0.5 * rankdata(cb_preds)/len(cb_preds)
```

---
*Upvote if useful 🙌 | [XGBoost GPU Baseline](https://www.kaggle.com/code/simarbirsinghsandhu/xgboost-gpu-fe-encoding) | [Full EDA](https://www.kaggle.com/code/simarbirsinghsandhu/s6e5-detailed-eda-original-dataset-analysis)*